# Notebook 3 — Single-Period RTS-GMLC DC Optimal Power Flow

## Objective

Construct, solve, and validate a single-period DC optimal power flow
for the full RTS-GMLC benchmark system.

The model will contain:

- 73 buses;
- 120 physical transmission branches;
- 158 generators;
- bus-level demand;
- generator operating limits;
- DC transmission physics;
- branch thermal limits; and
- generation operating costs.

This notebook extends the three-bus DC-OPF developed in Notebook 2 to
the actual RTS-GMLC network.

The primary goal is model correctness and validation before introducing
chronology, data-center demand, flexible workloads, or reconductoring
experiments.


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

RTS_PATH = (
    Path.home()
    / "RTS-GMLC"
    / "RTS_Data"
    / "SourceData"
)

bus = pd.read_csv(
    RTS_PATH / "bus.csv"
)

branch = pd.read_csv(
    RTS_PATH / "branch.csv"
)

gen = pd.read_csv(
    RTS_PATH / "gen.csv"
)

print("Buses:", bus.shape)
print("Branches:", branch.shape)
print("Generators:", gen.shape)

In [ ]:
print(gen.columns.tolist())

In [ ]:
gen.head()

In [ ]:
generators_per_bus = (
    gen.groupby("Bus ID")
    .size()
    .sort_values(ascending=False)
)

generators_per_bus.head(15)

In [ ]:
print(
    "Buses containing generation:",
    gen["Bus ID"].nunique()
)

print(
    "Maximum generators at one bus:",
    generators_per_bus.max()
)

In [ ]:
generators_per_bus[
    generators_per_bus > 1
].head(20)

In [ ]:
bus_ids = set(bus["Bus ID"])

generator_bus_ids = set(
    gen["Bus ID"]
)

invalid_generator_buses = (
    generator_bus_ids
    - bus_ids
)

print(
    "Generator buses not found in bus.csv:",
    invalid_generator_buses
)

In [ ]:
print(
    "Generators:",
    len(gen)
)

print(
    "Unique generator IDs:",
    gen["GEN UID"].nunique()
)

In [ ]:
for col in gen.columns:
    print(col)

In [ ]:
candidate_cost_columns = [
    col
    for col in gen.columns
    if any(
        keyword in col.lower()
        for keyword in [
            "cost",
            "heat",
            "fuel",
            "price",
            "hr",
            "vom"
        ]
    )
]

candidate_cost_columns

In [ ]:
display_columns = (
    ["GEN UID", "Bus ID"]
    + candidate_cost_columns
)

gen[
    display_columns
].head(20)

In [ ]:
output_cols = [
    "GEN UID",
    "Fuel",
    "PMax MW",
    "PMin MW",
    "Output_pct_0",
    "Output_pct_1",
    "Output_pct_2",
    "Output_pct_3",
    "Output_pct_4",
    "HR_avg_0",
    "HR_incr_1",
    "HR_incr_2",
    "HR_incr_3",
    "HR_incr_4",
    "Fuel Price $/MMBTU",
    "VOM"
]

gen[output_cols].head(30)

In [ ]:
for col in [
    "Output_pct_0",
    "Output_pct_1",
    "Output_pct_2",
    "Output_pct_3",
    "Output_pct_4",
    "HR_avg_0",
    "HR_incr_1",
    "HR_incr_2",
    "HR_incr_3",
    "HR_incr_4",
]:
    print(
        f"{col}: "
        f"{gen[col].notna().sum()} non-null values"
    )

In [ ]:
gen[
    gen["Output_pct_4"].notna()
][
    output_cols
].head(30)

In [ ]:
gen[
    [
        "GEN UID",
        "Fuel",
        "PMax MW",
        "PMin MW",
        "Output_pct_0",
        "Output_pct_1",
        "Output_pct_2",
        "Output_pct_3",
        "Output_pct_4",
    ]
].head(25)

In [ ]:
gen[
    [
        "Output_pct_0",
        "Output_pct_1",
        "Output_pct_2",
        "Output_pct_3",
        "Output_pct_4",
    ]
].describe()

In [ ]:
cost_summary_by_fuel = (
    gen.groupby("Fuel")
    .agg(
        generators=("GEN UID", "count"),
        total_capacity=("PMax MW", "sum"),
        fuel_price=("Fuel Price $/MMBTU", "median"),
        median_hr0=("HR_avg_0", "median"),
        median_vom=("VOM", "median")
    )
    .sort_values(
        "total_capacity",
        ascending=False
    )
)

cost_summary_by_fuel

In [ ]:
gen[
    gen["Fuel"].isin(
        [
            "Wind",
            "Solar",
            "Hydro",
            "Storage",
            "Sync_Cond"
        ]
    )
][
    [
        "GEN UID",
        "Fuel",
        "PMax MW",
        "PMin MW",
        "Fuel Price $/MMBTU",
        "HR_avg_0",
        "HR_incr_1",
        "VOM"
    ]
].head(50)

## 9. Generator Piecewise-Linear Cost Construction

RTS-GMLC represents generator production economics using **output breakpoints, heat rates, fuel prices, and variable operation and maintenance (VOM) costs**.

For generator $g$, each output breakpoint is calculated as

$$
P_{g,k}
=
\text{OutputPct}_{g,k} P_g^{\max}.
$$

For example, if a generator has

$$
P_g^{\max}=20\text{ MW}
$$

and its output fractions are

$$
(0.4,\;0.6,\;0.8,\;1.0),
$$

then its physical output breakpoints are

$$
(8,\;12,\;16,\;20)\text{ MW}.
$$

### Heat-rate representation

The first interval, from zero to the first breakpoint, uses the average heat rate

$$
HR_{g,0}=HR_{\text{avg},0}.
$$

Subsequent intervals use the corresponding incremental heat rates

$$
HR_{g,k}=HR_{\text{incr},k},
\qquad k\ge1.
$$

Because the heat-rate data are reported in BTU/kWh,

$$
1000\text{ BTU/kWh}
=
1\text{ MMBTU/MWh}.
$$

Therefore, the marginal operating cost associated with segment $k$ is

$$
c_{g,k}
=
\frac{HR_{g,k}}{1000}
\cdot
FuelPrice_g
+
VOM_g.
$$

The output of a generator can then be decomposed into piecewise segments:

$$
P_g
=
\sum_k p_{g,k}.
$$

Each segment has its own capacity and marginal operating cost.

### Initial modeling assumptions

The first RTS-GMLC model will be a **single-period DC optimal power flow**, not a unit-commitment model.

Therefore:

- binary commitment decisions are not yet included;
- startup and shutdown costs are not yet included;
- minimum up/down times are not yet included;
- storage is excluded from active-power dispatch because a single-period model cannot enforce intertemporal state-of-charge conservation.

These simplifications allow the initial RTS-GMLC model to remain focused on validating network-constrained economic dispatch before chronological constraints are introduced.

In [ ]:
segment_rows = []

for _, row in gen.iterrows():

    gen_id = row["GEN UID"]
    fuel = row["Fuel"]
    pmax = float(row["PMax MW"])
    fuel_price = float(row["Fuel Price $/MMBTU"])
    vom = float(row["VOM"])

    # Storage cannot be modeled correctly in a
    # single-period dispatch without state of charge.
    if fuel == "Storage":
        continue

    previous_mw = 0.0

    # Segment 0 uses average heat rate.
    # Later segments use incremental heat rates.
    for k in range(5):

        output_col = f"Output_pct_{k}"

        if pd.isna(row[output_col]):
            continue

        output_fraction = float(
            row[output_col]
        )

        breakpoint_mw = (
            output_fraction * pmax
        )

        segment_width = (
            breakpoint_mw - previous_mw
        )

        # Ignore zero-width segments.
        if segment_width <= 1e-9:
            previous_mw = breakpoint_mw
            continue

        if k == 0:
            heat_rate = float(
                row["HR_avg_0"]
            )
        else:
            heat_rate = float(
                row[f"HR_incr_{k}"]
            )

        marginal_cost = (
            heat_rate / 1000
            * fuel_price
            + vom
        )

        segment_rows.append({
            "GEN UID": gen_id,
            "Bus ID": row["Bus ID"],
            "Fuel": fuel,
            "Segment": k,
            "Start MW": previous_mw,
            "End MW": breakpoint_mw,
            "Width MW": segment_width,
            "Heat Rate BTU/kWh": heat_rate,
            "Fuel Price $/MMBTU": fuel_price,
            "VOM $/MWh": vom,
            "Marginal Cost $/MWh": marginal_cost
        })

        previous_mw = breakpoint_mw


segments = pd.DataFrame(
    segment_rows
)

print(
    "Total cost segments:",
    len(segments)
)

segments.head(20)

In [ ]:
segments[
    segments["GEN UID"] == "101_CT_1"
]

In [ ]:
segment_capacity_check = (
    segments
    .groupby("GEN UID")["Width MW"]
    .sum()
)

non_storage = gen[
    gen["Fuel"] != "Storage"
].set_index("GEN UID")

capacity_comparison = pd.DataFrame({
    "Segment Capacity": segment_capacity_check,
    "PMax": non_storage["PMax MW"]
})

capacity_comparison[
    "Error"
] = (
    capacity_comparison["Segment Capacity"]
    -
    capacity_comparison["PMax"]
)

print(
    "Maximum capacity reconstruction error:",
    capacity_comparison["Error"]
    .abs()
    .max()
)

In [ ]:
segment_counts = (
    segments
    .groupby("GEN UID")
    .size()
    .value_counts()
    .sort_index()
)

segment_counts

In [ ]:
segment_cost_summary = (
    segments
    .groupby("Fuel")[
        "Marginal Cost $/MWh"
    ]
    .agg(
        ["count", "min", "median", "max"]
    )
    .sort_values("median")
)

segment_cost_summary

In [ ]:
cost_curve_rows = []

for gen_id, group in segments.groupby("GEN UID"):

    group = group.sort_values(
        "Segment"
    )

    cumulative_heat = 0.0
    cumulative_cost = 0.0

    for _, seg in group.iterrows():

        heat_rate_mmbtu_per_mwh = (
            seg["Heat Rate BTU/kWh"]
            / 1000
        )

        segment_heat = (
            heat_rate_mmbtu_per_mwh
            * seg["Width MW"]
        )

        segment_cost = (
            seg["Marginal Cost $/MWh"]
            * seg["Width MW"]
        )

        cumulative_heat += segment_heat
        cumulative_cost += segment_cost

        cost_curve_rows.append({
            "GEN UID": gen_id,
            "Fuel": seg["Fuel"],
            "Segment": seg["Segment"],
            "Output MW": seg["End MW"],
            "Cumulative Heat MMBTU/h":
                cumulative_heat,
            "Cumulative Cost $/h":
                cumulative_cost
        })


cost_curves = pd.DataFrame(
    cost_curve_rows
)

cost_curves.head(20)

In [ ]:
cost_curves[
    cost_curves["GEN UID"]
    == "101_CT_1"
]

In [ ]:
full_load_cost = (
    cost_curves
    .sort_values(
        ["GEN UID", "Output MW"]
    )
    .groupby("GEN UID")
    .tail(1)
    .copy()
)

full_load_cost[
    "Average Full-Load Cost $/MWh"
] = (
    full_load_cost[
        "Cumulative Cost $/h"
    ]
    /
    full_load_cost[
        "Output MW"
    ]
)

full_load_cost[
    [
        "GEN UID",
        "Fuel",
        "Output MW",
        "Cumulative Cost $/h",
        "Average Full-Load Cost $/MWh"
    ]
].head(30)

In [ ]:
full_load_cost_summary = (
    full_load_cost
    .groupby("Fuel")[
        "Average Full-Load Cost $/MWh"
    ]
    .agg(
        ["count", "min", "median", "max"]
    )
    .sort_values("median")
)

full_load_cost_summary

In [ ]:
# Start with every generator in gen.csv
gen_model = gen[
    [
        "GEN UID",
        "Bus ID",
        "Fuel",
        "PMax MW",
        "PMin MW"
    ]
].copy()

# Cost coefficients calculated from the heat-rate curves
calculated_costs = (
    full_load_cost[
        [
            "GEN UID",
            "Average Full-Load Cost $/MWh"
        ]
    ]
    .set_index("GEN UID")
    ["Average Full-Load Cost $/MWh"]
)

# Map them back to ALL 158 generators
gen_model["Marginal Cost $/MWh"] = (
    gen_model["GEN UID"]
    .map(calculated_costs)
)

gen_model[
    [
        "Fuel",
        "Marginal Cost $/MWh"
    ]
].groupby("Fuel").agg(
    ["count", "min", "median", "max"]
)

In [ ]:
missing_costs = gen_model[
    gen_model[
        "Marginal Cost $/MWh"
    ].isna()
]

print(
    "Generators without calculated cost:",
    len(missing_costs)
)

missing_costs[
    [
        "GEN UID",
        "Bus ID",
        "Fuel",
        "PMax MW",
        "PMin MW"
    ]
]

In [ ]:
missing_costs.groupby(
    "Fuel"
).agg(
    generators=("GEN UID", "count"),
    capacity=("PMax MW", "sum")
)

In [ ]:
zero_cost_fuels = [
    "Wind",
    "Solar"
]

zero_cost_mask = (
    gen_model["Fuel"].isin(zero_cost_fuels)
    &
    gen_model[
        "Marginal Cost $/MWh"
    ].isna()
)

gen_model.loc[
    zero_cost_mask,
    "Marginal Cost $/MWh"
] = 0.0

In [ ]:
gen_model[
    gen_model["Fuel"] == "Storage"
]

In [ ]:
# Final bookkeeping cost for storage.
# Storage dispatch itself will be disabled in Notebook 3.
storage_mask = (
    gen_model["Fuel"] == "Storage"
)

gen_model.loc[
    storage_mask,
    "Marginal Cost $/MWh"
] = 0.0

# Initial available capacity.
# This will later be replaced by chronological availability
# for variable renewable generators.
gen_model["Available PMax MW"] = (
    gen_model["PMax MW"].astype(float)
)

# Disable storage in the single-period model.
gen_model.loc[
    storage_mask,
    "Available PMax MW"
] = 0.0

print(
    "Remaining missing costs:",
    gen_model["Marginal Cost $/MWh"]
    .isna()
    .sum()
)

print(
    "Total generators:",
    len(gen_model)
)

print(
    "Total nameplate capacity:",
    gen_model["PMax MW"].sum()
)

print(
    "Currently dispatchable capacity:",
    gen_model["Available PMax MW"].sum()
)

## 12. Construction of the Peak-Hour Operating State

The first full RTS-GMLC DC-OPF will represent a single operating hour rather than the static maximum-load condition.

The selected hour is the maximum system-load observation in the DAY_AHEAD chronology:

$$
t^* = \text{2020-08-26 14:00}.
$$

At this hour, total modeled system demand is approximately

$$
D_{t^*}=8{,}191.84\text{ MW}.
$$

Bus-level demand is reconstructed from regional DAY_AHEAD load using the fixed within-area load shares developed in Notebook 1:

$$
D_{i,t}
=
s_iD_{a(i),t},
$$

where

$$
s_i
=
\frac{D_i^{static}}
{\sum_{j\in a(i)}D_j^{static}}.
$$

This preserves each area's chronological total exactly while assuming that the spatial distribution of load within each area remains proportional to the static RTS-GMLC bus-load distribution.

Variable renewable generation will use availability from the same chronological hour rather than static nameplate capacity.

In [ ]:
TS_PATH = (
    Path.home()
    / "RTS-GMLC"
    / "RTS_Data"
    / "timeseries_data_files"
)

load_file = (
    TS_PATH
    / "Load"
    / "DAY_AHEAD_regional_Load.csv"
)

regional_load = pd.read_csv(
    load_file
)

regional_load["Timestamp"] = pd.to_datetime(
    dict(
        year=regional_load["Year"],
        month=regional_load["Month"],
        day=regional_load["Day"]
    )
) + pd.to_timedelta(
    regional_load["Period"] - 1,
    unit="h"
)

regional_load[
    "System Load MW"
] = (
    regional_load["1"]
    + regional_load["2"]
    + regional_load["3"]
)

peak_idx = (
    regional_load["System Load MW"]
    .idxmax()
)

peak_row = regional_load.loc[
    peak_idx
]

peak_timestamp = peak_row["Timestamp"]

print(
    "Peak timestamp:",
    peak_timestamp
)

print(
    "Peak system load:",
    peak_row["System Load MW"]
)

print(
    "Area loads:"
)

print(
    peak_row[
        ["1", "2", "3"]
    ]
)

In [ ]:
print(
    bus[
        [
            "Bus ID",
            "Area",
            "MW Load"
        ]
    ].head(20)
)

In [ ]:
area_static_load = (
    bus.groupby("Area")[
        "MW Load"
    ].sum()
)

print(
    "Static load by area:"
)

print(area_static_load)

In [ ]:
bus_peak = bus[
    [
        "Bus ID",
        "Area",
        "MW Load"
    ]
].copy()

bus_peak[
    "Area Static Load MW"
] = (
    bus_peak["Area"]
    .map(area_static_load)
)

bus_peak[
    "Load Share"
] = (
    bus_peak["MW Load"]
    /
    bus_peak["Area Static Load MW"]
)

In [ ]:
print(
    "Load-share sums by area:"
)

print(
    bus_peak.groupby("Area")[
        "Load Share"
    ].sum()
)

In [ ]:
peak_area_load = {
    1: float(peak_row["1"]),
    2: float(peak_row["2"]),
    3: float(peak_row["3"])
}

bus_peak[
    "Peak Load MW"
] = (
    bus_peak.apply(
        lambda row:
        row["Load Share"]
        * peak_area_load[
            int(row["Area"])
        ],
        axis=1
    )
)

In [ ]:
print(
    "Reconstructed bus-level load:",
    bus_peak["Peak Load MW"].sum()
)

print(
    "Original regional system load:",
    peak_row["System Load MW"]
)

print(
    "Reconstruction error:",
    abs(
        bus_peak["Peak Load MW"].sum()
        -
        peak_row["System Load MW"]
    )
)

In [ ]:
bus_peak[
    [
        "Bus ID",
        "Area",
        "Peak Load MW"
    ]
].sort_values(
    "Peak Load MW",
    ascending=False
).head(15)

In [ ]:
print(
    "Buses with positive load:",
    (
        bus_peak["Peak Load MW"] > 0
    ).sum()
)

print(
    "Largest bus load:",
    bus_peak["Peak Load MW"].max()
)

In [ ]:
sync_mask = (
    gen_model["Fuel"] == "Sync_Cond"
)

gen_model.loc[
    sync_mask,
    "Marginal Cost $/MWh"
] = 0.0

print(
    "Remaining missing costs:",
    gen_model["Marginal Cost $/MWh"]
    .isna()
    .sum()
)

In [ ]:
pointers = pd.read_csv(
    RTS_PATH / "timeseries_pointers.csv"
)

da_pmax = pointers[
    (pointers["Simulation"] == "DAY_AHEAD")
    &
    (pointers["Category"] == "Generator")
    &
    (pointers["Parameter"] == "PMax MW")
].copy()

print(
    "DAY_AHEAD generator PMax pointers:",
    len(da_pmax)
)

print(
    "Unique time-series files:",
    da_pmax["Data File"].nunique()
)

print()

print(
    da_pmax.groupby(
        "Data File"
    ).size()
)

In [ ]:
da_pmax[
    [
        "Object",
        "Parameter",
        "Scaling Factor",
        "Data File"
    ]
].head(30)

In [ ]:
print(
    "Generator fuels represented "
    "by DAY_AHEAD PMax pointers:"
)

pointer_fuels = (
    da_pmax[
        ["Object"]
    ]
    .merge(
        gen[
            ["GEN UID", "Fuel"]
        ],
        left_on="Object",
        right_on="GEN UID",
        how="left"
    )
)

print(
    pointer_fuels.groupby("Fuel")
    .size()
)

In [ ]:
def load_rts_timeseries(relative_path):
    """
    Load an RTS-GMLC time-series CSV and construct
    its hourly timestamp column.
    """

    file_path = (
        Path.home()
        / "RTS-GMLC"
        / "RTS_Data"
        / relative_path
    )

    df = pd.read_csv(file_path)

    df["Timestamp"] = pd.to_datetime(
        dict(
            year=df["Year"],
            month=df["Month"],
            day=df["Day"]
        )
    ) + pd.to_timedelta(
        df["Period"] - 1,
        unit="h"
    )

    return df

In [ ]:
print(
    da_pmax["Data File"]
    .drop_duplicates()
    .tolist()
)

## 14. Peak-Hour Generator Availability

Generator capacity is not necessarily constant through time.

For conventional thermal generators, the initial single-period model uses the static RTS-GMLC maximum capacity:

$$
\overline P_{g,t^*}=P_g^{\max}.
$$

For generators with DAY_AHEAD time-series data, the static capacity is replaced by the availability reported for the selected operating hour:

$$
\overline P_{g,t^*}
=
P^{DA}_{g,t^*}.
$$

The selected hour is

$$
t^*=\text{2020-08-26 14:00}.
$$

This ensures that demand and time-dependent generation availability are taken from the same chronological system state.

The resulting quantity

$$
\overline P_{g,t^*}
$$

is an upper bound on dispatch, not required generation. The optimization remains free to curtail generation:

$$
0\le P_g\le\overline P_{g,t^*}.
$$

Storage remains unavailable in the single-period model because no intertemporal state-of-charge constraint is present.

In [ ]:
def load_pointer_timeseries(relative_path):
    """
    Load an RTS-GMLC time-series file using a path
    stored in timeseries_pointers.csv.
    """

    file_path = (
        RTS_PATH / relative_path
    ).resolve()

    df = pd.read_csv(file_path)

    df["Timestamp"] = pd.to_datetime(
        dict(
            year=df["Year"],
            month=df["Month"],
            day=df["Day"]
        )
    ) + pd.to_timedelta(
        df["Period"] - 1,
        unit="h"
    )

    return df

In [ ]:
unique_da_files = (
    da_pmax["Data File"]
    .drop_duplicates()
    .tolist()
)

for file in unique_da_files:

    ts = load_pointer_timeseries(file)

    match = ts[
        ts["Timestamp"] == peak_timestamp
    ]

    print(
        file,
        "| rows:", len(ts),
        "| peak matches:", len(match)
    )

In [ ]:
peak_availability = (
    gen_model
    .set_index("GEN UID")[
        "PMax MW"
    ]
    .astype(float)
    .to_dict()
)

In [ ]:
for file in unique_da_files:

    ts = load_pointer_timeseries(file)

    peak_ts_row = ts.loc[
        ts["Timestamp"] == peak_timestamp
    ]

    assert len(peak_ts_row) == 1

    peak_ts_row = peak_ts_row.iloc[0]

    file_pointers = da_pmax[
        da_pmax["Data File"] == file
    ]

    for _, pointer in file_pointers.iterrows():

        gen_id = pointer["Object"]

        if gen_id not in ts.columns:
            raise ValueError(
                f"{gen_id} not found in {file}"
            )

        peak_availability[gen_id] = float(
            peak_ts_row[gen_id]
        )

In [ ]:
gen_model[
    "Available PMax MW"
] = (
    gen_model["GEN UID"]
    .map(peak_availability)
)

# Storage remains disabled.
gen_model.loc[
    gen_model["Fuel"] == "Storage",
    "Available PMax MW"
] = 0.0

In [ ]:
print(
    "Missing availability values:",
    gen_model["Available PMax MW"]
    .isna()
    .sum()
)

print(
    "Negative availability values:",
    (
        gen_model["Available PMax MW"] < 0
    ).sum()
)

In [ ]:
gen_model[
    "Availability / Nameplate"
] = np.where(
    gen_model["PMax MW"] > 0,
    gen_model["Available PMax MW"]
    / gen_model["PMax MW"],
    np.nan
)

availability_violations = gen_model[
    gen_model["Available PMax MW"]
    >
    gen_model["PMax MW"] + 1e-6
]

print(
    "Generators above static PMax:",
    len(availability_violations)
)

availability_violations[
    [
        "GEN UID",
        "Fuel",
        "PMax MW",
        "Available PMax MW",
        "Availability / Nameplate"
    ]
]

In [ ]:
peak_capacity_by_fuel = (
    gen_model
    .groupby("Fuel")
    .agg(
        Generators=("GEN UID", "count"),
        Nameplate_MW=("PMax MW", "sum"),
        Available_MW=(
            "Available PMax MW",
            "sum"
        )
    )
)

peak_capacity_by_fuel[
    "Availability_Fraction"
] = (
    peak_capacity_by_fuel["Available_MW"]
    /
    peak_capacity_by_fuel["Nameplate_MW"]
)

peak_capacity_by_fuel

In [ ]:
total_peak_available = (
    gen_model[
        "Available PMax MW"
    ].sum()
)

total_peak_demand = (
    bus_peak[
        "Peak Load MW"
    ].sum()
)

capacity_surplus = (
    total_peak_available
    -
    total_peak_demand
)

print(
    f"Peak demand: "
    f"{total_peak_demand:,.2f} MW"
)

print(
    f"Available generation: "
    f"{total_peak_available:,.2f} MW"
)

print(
    f"Aggregate capacity surplus: "
    f"{capacity_surplus:,.2f} MW"
)

print(
    f"Available capacity / demand: "
    f"{total_peak_available / total_peak_demand:.3f}x"
)

In [ ]:
variable_fuels = [
    "Wind",
    "Solar",
    "Hydro"
]

gen_model[
    gen_model["Fuel"].isin(
        variable_fuels
    )
][
    [
        "GEN UID",
        "Bus ID",
        "Fuel",
        "PMax MW",
        "Available PMax MW",
        "Availability / Nameplate"
    ]
].sort_values(
    "PMax MW",
    ascending=False
).head(25)

## 15. Full RTS-GMLC Single-Period DC Optimal Power Flow

The audited network, generator, cost, demand, and availability data can now be combined into the first full RTS-GMLC optimization model.

The model contains:

- 73 buses;
- 120 physical transmission branches;
- 158 generator objects;
- chronological peak-hour demand of approximately 8,191.84 MW;
- generator-specific peak-hour availability;
- generator-specific linear production-cost approximations.

### Objective

The optimization minimizes total hourly generation cost:

$$
\min \sum_{g \in G} c_g P_g,
$$

where $P_g$ is generator dispatch and $c_g$ is its baseline marginal-cost approximation.

### Generator limits

Each generator is constrained by its availability during the selected hour:

$$
0 \le P_g \le \overline{P}_{g,t^*}.
$$

### DC power flow

For each transmission branch $\ell$ connecting buses $i$ and $j$,

$$
f_\ell
=
\frac{\theta_i-\theta_j}{X_\ell}S_{\text{base}}.
$$

### Transmission limits

Each physical branch must respect its thermal rating:

$$
-F_\ell^{\max}
\le
f_\ell
\le
F_\ell^{\max}.
$$

### Nodal power balance

At every bus $i$,

$$
\sum_{g:n(g)=i}P_g-D_i
=
\sum_{\ell\in\delta^+(i)}f_\ell
-
\sum_{\ell\in\delta^-(i)}f_\ell.
$$

### Reference angle

Because only voltage-angle differences affect DC power flow, one bus angle is fixed:

$$
\theta_{101}=0.
$$

This model is a linear program and represents network-constrained economic dispatch for one RTS-GMLC operating hour.

In [ ]:
# Sets
BUS_IDS = bus["Bus ID"].astype(int).tolist()
GEN_IDS = gen_model["GEN UID"].tolist()
LINE_IDS = branch["UID"].tolist()

print("Buses:", len(BUS_IDS))
print("Generators:", len(GEN_IDS))
print("Branches:", len(LINE_IDS))

In [ ]:
gen_bus = (
    gen_model
    .set_index("GEN UID")["Bus ID"]
    .astype(int)
    .to_dict()
)

gen_pmax = (
    gen_model
    .set_index("GEN UID")["Available PMax MW"]
    .astype(float)
    .to_dict()
)

gen_cost = (
    gen_model
    .set_index("GEN UID")["Marginal Cost $/MWh"]
    .astype(float)
    .to_dict()
)

In [ ]:
print(
    "Reconstructed bus-level load:",
    bus_peak["Peak Load MW"].sum()
)

print(
    "Original regional system load:",
    peak_row["System Load MW"]
)

print(
    "Reconstruction error:",
    abs(
        bus_peak["Peak Load MW"].sum()
        -
        peak_row["System Load MW"]
    )
)

In [ ]:
line_from = (
    branch
    .set_index("UID")["From Bus"]
    .astype(int)
    .to_dict()
)

line_to = (
    branch
    .set_index("UID")["To Bus"]
    .astype(int)
    .to_dict()
)

line_x = (
    branch
    .set_index("UID")["X"]
    .astype(float)
    .to_dict()
)

line_limit = (
    branch
    .set_index("UID")["Cont Rating"]
    .astype(float)
    .to_dict()
)

In [ ]:
gens_at_bus = {
    b: [] for b in BUS_IDS
}

for g in GEN_IDS:
    gens_at_bus[
        gen_bus[g]
    ].append(g)

In [ ]:
print(
    "Generators mapped:",
    sum(
        len(v)
        for v in gens_at_bus.values()
    )
)

print(
    "Maximum generators at one bus:",
    max(
        len(v)
        for v in gens_at_bus.values()
    )
)

In [ ]:
lines_from_bus = {
    b: [] for b in BUS_IDS
}

lines_to_bus = {
    b: [] for b in BUS_IDS
}

for l in LINE_IDS:

    lines_from_bus[
        line_from[l]
    ].append(l)

    lines_to_bus[
        line_to[l]
    ].append(l)

In [ ]:
print(
    "Outgoing branch references:",
    sum(
        len(v)
        for v in lines_from_bus.values()
    )
)

print(
    "Incoming branch references:",
    sum(
        len(v)
        for v in lines_to_bus.values()
    )
)

In [ ]:
demand = (
    bus_peak
    .set_index("Bus ID")["Peak Load MW"]
    .astype(float)
    .to_dict()
)

In [ ]:
assert len(demand) == 73
assert set(demand.keys()) == set(BUS_IDS)
assert abs(sum(demand.values()) - 8191.835957) < 1e-6

print("Demand dictionary validated.")

In [ ]:
import pyomo.environ as pyo

model = pyo.ConcreteModel()

model.BUSES = pyo.Set(
    initialize=BUS_IDS
)

model.GENS = pyo.Set(
    initialize=GEN_IDS
)

model.LINES = pyo.Set(
    initialize=LINE_IDS
)

In [ ]:
model.P = pyo.Var(
    model.GENS,
    domain=pyo.NonNegativeReals
)

model.theta = pyo.Var(
    model.BUSES,
    domain=pyo.Reals
)

model.flow = pyo.Var(
    model.LINES,
    domain=pyo.Reals
)

In [ ]:
def generator_limit_rule(model, g):

    return (
        model.P[g]
        <= gen_pmax[g]
    )


model.generator_limit = pyo.Constraint(
    model.GENS,
    rule=generator_limit_rule
)

In [ ]:
model.reference_angle = pyo.Constraint(
    expr=model.theta[101] == 0
)

In [ ]:
S_BASE = 100.0

In [ ]:
def dc_flow_rule(model, l):

    i = line_from[l]
    j = line_to[l]

    return (
        model.flow[l]
        ==
        (
            model.theta[i]
            -
            model.theta[j]
        )
        / line_x[l]
        * S_BASE
    )


model.dc_flow = pyo.Constraint(
    model.LINES,
    rule=dc_flow_rule
)

In [ ]:
def line_limit_rule(model, l):

    return pyo.inequality(
        -line_limit[l],
        model.flow[l],
        line_limit[l]
    )


model.line_limit = pyo.Constraint(
    model.LINES,
    rule=line_limit_rule
)

In [ ]:
def nodal_balance_rule(model, b):

    generation = sum(
        model.P[g]
        for g in gens_at_bus[b]
    )

    outgoing = sum(
        model.flow[l]
        for l in lines_from_bus[b]
    )

    incoming = sum(
        model.flow[l]
        for l in lines_to_bus[b]
    )

    return (
        generation
        - demand[b]
        ==
        outgoing
        - incoming
    )


model.nodal_balance = pyo.Constraint(
    model.BUSES,
    rule=nodal_balance_rule
)

In [ ]:
model.objective = pyo.Objective(
    expr=sum(
        gen_cost[g]
        * model.P[g]
        for g in GEN_IDS
    ),
    sense=pyo.minimize
)

In [ ]:
model.dual = pyo.Suffix(
    direction=pyo.Suffix.IMPORT
)

In [ ]:
solver = pyo.SolverFactory(
    "highs"
)

results = solver.solve(
    model,
    tee=False
)

print(
    "Solver status:",
    results.solver.status
)

print(
    "Termination condition:",
    results.solver.termination_condition
)

## 16. Independent Validation of the RTS-GMLC DC-OPF

HiGHS reports an optimal solution, but solver optimality alone does not establish that the implemented model correctly represents the intended power-system equations.

The solution is therefore independently checked for:

1. system-wide power balance;
2. generator availability limits;
3. transmission thermal limits;
4. DC branch-flow equations;
5. nodal power balance at all 73 buses.

These checks are calculated directly from the optimized variable values rather than relying only on the solver termination condition.

In [ ]:
dispatch = {
    g: pyo.value(model.P[g])
    for g in GEN_IDS
}

angles = {
    b: pyo.value(model.theta[b])
    for b in BUS_IDS
}

flows = {
    l: pyo.value(model.flow[l])
    for l in LINE_IDS
}

objective_value = pyo.value(
    model.objective
)

print(
    f"Objective value: "
    f"${objective_value:,.2f}/h"
)

In [ ]:
total_generation = sum(
    dispatch.values()
)

total_demand = sum(
    demand.values()
)

system_balance_error = (
    total_generation
    - total_demand
)

print(
    f"Total generation: "
    f"{total_generation:,.6f} MW"
)

print(
    f"Total demand: "
    f"{total_demand:,.6f} MW"
)

print(
    f"System balance error: "
    f"{system_balance_error:.12e} MW"
)

In [ ]:
GEN_TOL = 1e-6

generator_violations = []

for g in GEN_IDS:

    p = dispatch[g]
    pmax = gen_pmax[g]

    if (
        p < -GEN_TOL
        or
        p > pmax + GEN_TOL
    ):
        generator_violations.append(
            {
                "GEN UID": g,
                "Dispatch MW": p,
                "Available PMax MW": pmax
            }
        )

print(
    "Generator-limit violations:",
    len(generator_violations)
)

generator_violations[:10]

In [ ]:
LINE_TOL = 1e-6

line_violations = []

for l in LINE_IDS:

    f = flows[l]
    limit = line_limit[l]

    if abs(f) > limit + LINE_TOL:

        line_violations.append(
            {
                "Line": l,
                "Flow MW": f,
                "Limit MW": limit
            }
        )

print(
    "Thermal-limit violations:",
    len(line_violations)
)

line_violations[:10]

In [ ]:
dc_flow_errors = {}

for l in LINE_IDS:

    i = line_from[l]
    j = line_to[l]

    reconstructed_flow = (
        (
            angles[i]
            -
            angles[j]
        )
        /
        line_x[l]
        *
        S_BASE
    )

    dc_flow_errors[l] = (
        flows[l]
        -
        reconstructed_flow
    )

max_dc_flow_error = max(
    abs(error)
    for error in dc_flow_errors.values()
)

print(
    "Maximum DC-flow equation error:",
    f"{max_dc_flow_error:.12e} MW"
)

In [ ]:
nodal_errors = {}

for b in BUS_IDS:

    generation_at_bus = sum(
        dispatch[g]
        for g in gens_at_bus[b]
    )

    outgoing_flow = sum(
        flows[l]
        for l in lines_from_bus[b]
    )

    incoming_flow = sum(
        flows[l]
        for l in lines_to_bus[b]
    )

    nodal_errors[b] = (
        generation_at_bus
        - demand[b]
        - (
            outgoing_flow
            - incoming_flow
        )
    )

max_nodal_error = max(
    abs(error)
    for error in nodal_errors.values()
)

print(
    "Maximum nodal-balance error:",
    f"{max_nodal_error:.12e} MW"
)

In [ ]:
worst_bus = max(
    nodal_errors,
    key=lambda b: abs(nodal_errors[b])
)

print(
    "Bus with largest balance error:",
    worst_bus
)

print(
    "Error:",
    nodal_errors[worst_bus]
)

In [ ]:
validation_summary = {
    "System balance error MW":
        abs(system_balance_error),

    "Generator violations":
        len(generator_violations),

    "Thermal-limit violations":
        len(line_violations),

    "Max DC-flow error MW":
        max_dc_flow_error,

    "Max nodal-balance error MW":
        max_nodal_error
}

pd.Series(
    validation_summary,
    name="Validation Result"
)

Rebuilding the Pyomo model from scratch


In [ ]:
demand = (
    bus_peak
    .set_index("Bus ID")["Peak Load MW"]
    .astype(float)
    .to_dict()
)

assert len(demand) == 73
assert set(demand.keys()) == set(BUS_IDS)
assert abs(
    sum(demand.values())
    - 8191.835957
) < 1e-6

print("Demand dictionary validated.")
print(
    "Total demand:",
    sum(demand.values())
)

In [ ]:
import pyomo.environ as pyo

# ============================================================
# FRESH MODEL
# ============================================================

model = pyo.ConcreteModel()

# Sets
model.BUSES = pyo.Set(
    initialize=BUS_IDS
)

model.GENS = pyo.Set(
    initialize=GEN_IDS
)

model.LINES = pyo.Set(
    initialize=LINE_IDS
)


# ============================================================
# VARIABLES
# ============================================================

model.P = pyo.Var(
    model.GENS,
    domain=pyo.NonNegativeReals
)

model.theta = pyo.Var(
    model.BUSES,
    domain=pyo.Reals
)

model.flow = pyo.Var(
    model.LINES,
    domain=pyo.Reals
)


# ============================================================
# GENERATOR LIMITS
# ============================================================

def generator_limit_rule(model, g):
    return (
        model.P[g]
        <= gen_pmax[g]
    )

model.generator_limit = pyo.Constraint(
    model.GENS,
    rule=generator_limit_rule
)


# ============================================================
# REFERENCE ANGLE
# ============================================================

model.reference_angle = pyo.Constraint(
    expr=model.theta[101] == 0
)


# ============================================================
# DC POWER FLOW
# ============================================================

S_BASE = 100.0

def dc_flow_rule(model, l):

    i = line_from[l]
    j = line_to[l]

    return (
        model.flow[l]
        ==
        (
            model.theta[i]
            - model.theta[j]
        )
        / line_x[l]
        * S_BASE
    )

model.dc_flow = pyo.Constraint(
    model.LINES,
    rule=dc_flow_rule
)


# ============================================================
# TRANSMISSION LIMITS
# ============================================================

def line_limit_rule(model, l):

    return pyo.inequality(
        -line_limit[l],
        model.flow[l],
        line_limit[l]
    )

model.line_limit = pyo.Constraint(
    model.LINES,
    rule=line_limit_rule
)


# ============================================================
# NODAL POWER BALANCE
# ============================================================

def nodal_balance_rule(model, b):

    generation = sum(
        model.P[g]
        for g in gens_at_bus[b]
    )

    outgoing = sum(
        model.flow[l]
        for l in lines_from_bus[b]
    )

    incoming = sum(
        model.flow[l]
        for l in lines_to_bus[b]
    )

    return (
        generation
        - demand[b]
        ==
        outgoing
        - incoming
    )

model.nodal_balance = pyo.Constraint(
    model.BUSES,
    rule=nodal_balance_rule
)


# ============================================================
# OBJECTIVE
# ============================================================

model.objective = pyo.Objective(
    expr=sum(
        gen_cost[g]
        * model.P[g]
        for g in GEN_IDS
    ),
    sense=pyo.minimize
)


# ============================================================
# DUALS
# ============================================================

model.dual = pyo.Suffix(
    direction=pyo.Suffix.IMPORT
)


# ============================================================
# MODEL CONSTRUCTION CHECKS
# ============================================================

print(
    "Generator constraints:",
    len(model.generator_limit)
)

print(
    "DC-flow constraints:",
    len(model.dc_flow)
)

print(
    "Line-limit constraints:",
    len(model.line_limit)
)

print(
    "Nodal-balance constraints:",
    len(model.nodal_balance)
)

print(
    "Reference-angle constraints:",
    len(model.reference_angle)
)

In [ ]:
print(
    model.nodal_balance[101].expr
)

In [ ]:
solver = pyo.SolverFactory(
    "highs"
)

results = solver.solve(
    model,
    tee=False
)

print(
    "Solver status:",
    results.solver.status
)

print(
    "Termination condition:",
    results.solver.termination_condition
)

In [ ]:
quick_generation = sum(
    pyo.value(model.P[g])
    for g in GEN_IDS
)

quick_demand = sum(
    demand.values()
)

print(
    f"Generation: {quick_generation:,.6f} MW"
)

print(
    f"Demand:     {quick_demand:,.6f} MW"
)

print(
    f"Difference: "
    f"{quick_generation - quick_demand:.12e} MW"
)

print(
    f"Objective: "
    f"${pyo.value(model.objective):,.2f}/h"
)

### 16.8 Validation Conclusion

The corrected RTS-GMLC peak-hour DC-OPF solved to optimality and independently satisfied all tested physical and operational constraints within numerical tolerance.

System-wide generation equals demand, all generator availability limits and transmission thermal limits are respected, and independently reconstructed DC-flow and nodal-balance equations agree with the optimized solution.

Therefore, the model is accepted as the validated baseline single-period DC-OPF for subsequent analysis.

The baseline objective value is approximately

$$
C^* = \$130{,}429.23/\text{h}.
$$

This value represents modeled generation cost under the simplified linear generator-cost approximation developed above. It should not be interpreted as an observed historical market cost.

## 17. Baseline Peak-Hour Dispatch Analysis

After validating the optimization model, the baseline solution is examined to understand how the RTS-GMLC system serves peak-hour demand.

The analysis focuses on:

- generation dispatch by technology;
- utilization of available generation;
- renewable curtailment;
- transmission utilization;
- binding transmission constraints;
- and the economic value of transmission capacity.

These results characterize the baseline operating state before introducing either data-center flexibility or reconductoring.

In [ ]:
dispatch_results = gen_model[
    [
        "GEN UID",
        "Bus ID",
        "Fuel",
        "PMax MW",
        "Available PMax MW",
        "Marginal Cost $/MWh"
    ]
].copy()

dispatch_results[
    "Dispatch MW"
] = (
    dispatch_results["GEN UID"]
    .map(dispatch)
)

dispatch_results[
    "Unused Available MW"
] = (
    dispatch_results["Available PMax MW"]
    -
    dispatch_results["Dispatch MW"]
)

dispatch_results[
    "Utilization"
] = np.where(
    dispatch_results["Available PMax MW"] > 0,
    dispatch_results["Dispatch MW"]
    /
    dispatch_results["Available PMax MW"],
    np.nan
)

In [ ]:
dispatch_results.sort_values(
    "Dispatch MW",
    ascending=False
).head(20)

In [ ]:
dispatch_by_fuel = (
    dispatch_results
    .groupby("Fuel")
    .agg(
        Generators=("GEN UID", "count"),
        Available_MW=("Available PMax MW", "sum"),
        Dispatch_MW=("Dispatch MW", "sum")
    )
)

dispatch_by_fuel[
    "Unused_Available_MW"
] = (
    dispatch_by_fuel["Available_MW"]
    -
    dispatch_by_fuel["Dispatch_MW"]
)

dispatch_by_fuel[
    "Utilization"
] = np.where(
    dispatch_by_fuel["Available_MW"] > 0,
    dispatch_by_fuel["Dispatch_MW"]
    /
    dispatch_by_fuel["Available_MW"],
    np.nan
)

dispatch_by_fuel.sort_values(
    "Dispatch_MW",
    ascending=False
)

In [ ]:
print(
    "Dispatch by fuel total:",
    dispatch_by_fuel[
        "Dispatch_MW"
    ].sum()
)

print(
    "System demand:",
    total_demand
)

In [ ]:
renewable_fuels = [
    "Hydro",
    "Solar",
    "Wind"
]

renewable_results = (
    dispatch_results[
        dispatch_results["Fuel"].isin(
            renewable_fuels
        )
    ]
    .groupby("Fuel")
    .agg(
        Available_MW=(
            "Available PMax MW",
            "sum"
        ),
        Dispatch_MW=(
            "Dispatch MW",
            "sum"
        )
    )
)

renewable_results[
    "Curtailed_MW"
] = (
    renewable_results["Available_MW"]
    -
    renewable_results["Dispatch_MW"]
)

renewable_results[
    "Curtailment_Rate"
] = (
    renewable_results["Curtailed_MW"]
    /
    renewable_results["Available_MW"]
)

renewable_results

In [ ]:
line_results = branch[
    [
        "UID",
        "From Bus",
        "To Bus",
        "X",
        "Cont Rating"
    ]
].copy()

line_results[
    "Flow MW"
] = (
    line_results["UID"]
    .map(flows)
)

line_results[
    "Absolute Flow MW"
] = (
    line_results["Flow MW"]
    .abs()
)

line_results[
    "Utilization"
] = (
    line_results["Absolute Flow MW"]
    /
    line_results["Cont Rating"]
)

line_results[
    "Remaining Capacity MW"
] = (
    line_results["Cont Rating"]
    -
    line_results["Absolute Flow MW"]
)

In [ ]:
line_results.sort_values(
    "Utilization",
    ascending=False
).head(20)

In [ ]:
binding_lines = line_results[
    line_results["Utilization"]
    >= 0.999
].sort_values(
    "Utilization",
    ascending=False
)

near_binding_lines = line_results[
    line_results["Utilization"]
    >= 0.90
].sort_values(
    "Utilization",
    ascending=False
)

print(
    "Binding lines:",
    len(binding_lines)
)

print(
    "Lines >= 90% utilization:",
    len(near_binding_lines)
)

binding_lines

In [ ]:
line_duals = {}

for l in LINE_IDS:
    line_duals[l] = (
        model.dual[
            model.line_limit[l]
        ]
        if model.line_limit[l]
        in model.dual
        else np.nan
    )

line_results[
    "Line Limit Dual"
] = (
    line_results["UID"]
    .map(line_duals)
)

line_results.sort_values(
    "Utilization",
    ascending=False
)[
    [
        "UID",
        "From Bus",
        "To Bus",
        "Flow MW",
        "Cont Rating",
        "Utilization",
        "Line Limit Dual"
    ]
].head(20)

In [ ]:
print(
    "Reconstructed bus-level load:",
    bus_peak["Peak Load MW"].sum()
)

print(
    "Original regional system load:",
    peak_row["System Load MW"]
)

print(
    "Reconstruction error:",
    abs(
        bus_peak["Peak Load MW"].sum()
        -
        peak_row["System Load MW"]
    )
)

## 18. Peak-Hour Load-Stress Experiment

The validated peak-hour baseline contains no binding transmission thermal constraints, although Line A27 operates at approximately 99.25% of its 500 MW rating.

A controlled load-stress experiment is therefore used to determine how close the system is to economically meaningful transmission congestion.

For each stress level $\alpha$, baseline bus demand is scaled proportionally:

$$
D_i(\alpha)
=
(1+\alpha)D_i^0,
$$

where $D_i^0$ is the validated peak-hour bus demand.

The optimization is re-solved independently at each stress level while holding constant:

- network topology;
- transmission ratings;
- generator costs;
- peak-hour generator availability;
- storage treatment.

The experiment measures how optimal dispatch, operating cost, transmission utilization, and binding constraints change as system demand increases.

This is a diagnostic sensitivity experiment rather than a representation of actual future load growth.

In [ ]:
stress_levels = [
    0.000,
    0.005,
    0.010,
    0.020,
    0.050
]

for alpha in stress_levels:
    print(
        f"{alpha:6.1%} -> "
        f"{total_demand * (1 + alpha):,.2f} MW"
    )

In [ ]:
def solve_rts_opf(
    demand_scenario,
    gen_pmax_scenario,
    line_limit_scenario=None
):
    """
    Solve one single-period RTS-GMLC DC-OPF.

    Parameters
    ----------
    demand_scenario : dict
        Bus-level demand in MW.

    gen_pmax_scenario : dict
        Generator-specific available PMax in MW.

    line_limit_scenario : dict, optional
        Branch thermal limits in MW.
        If None, baseline line limits are used.

    Returns
    -------
    model : Pyomo ConcreteModel
        Solved optimization model.

    results : SolverResults
        HiGHS solver results.
    """

    if line_limit_scenario is None:
        line_limit_scenario = line_limit

    m = pyo.ConcreteModel()

    # ----------------------------
    # Sets
    # ----------------------------

    m.BUSES = pyo.Set(
        initialize=BUS_IDS
    )

    m.GENS = pyo.Set(
        initialize=GEN_IDS
    )

    m.LINES = pyo.Set(
        initialize=LINE_IDS
    )

    # ----------------------------
    # Variables
    # ----------------------------

    m.P = pyo.Var(
        m.GENS,
        domain=pyo.NonNegativeReals
    )

    m.theta = pyo.Var(
        m.BUSES,
        domain=pyo.Reals
    )

    m.flow = pyo.Var(
        m.LINES,
        domain=pyo.Reals
    )

    # ----------------------------
    # Generator limits
    # ----------------------------

    def generator_limit_rule(m, g):
        return (
            m.P[g]
            <= gen_pmax_scenario[g]
        )

    m.generator_limit = pyo.Constraint(
        m.GENS,
        rule=generator_limit_rule
    )

    # ----------------------------
    # Reference angle
    # ----------------------------

    m.reference_angle = pyo.Constraint(
        expr=m.theta[101] == 0
    )

    # ----------------------------
    # DC power flow
    # ----------------------------

    def dc_flow_rule(m, l):

        i = line_from[l]
        j = line_to[l]

        return (
            m.flow[l]
            ==
            (
                m.theta[i]
                -
                m.theta[j]
            )
            / line_x[l]
            * S_BASE
        )

    m.dc_flow = pyo.Constraint(
        m.LINES,
        rule=dc_flow_rule
    )

    # ----------------------------
    # Separate line limits
    # ----------------------------

    def line_upper_rule(m, l):
        return (
            m.flow[l]
            <= line_limit_scenario[l]
        )

    def line_lower_rule(m, l):
        return (
            -m.flow[l]
            <= line_limit_scenario[l]
        )

    m.line_upper = pyo.Constraint(
        m.LINES,
        rule=line_upper_rule
    )

    m.line_lower = pyo.Constraint(
        m.LINES,
        rule=line_lower_rule
    )

    # ----------------------------
    # Nodal balance
    # ----------------------------

    def nodal_balance_rule(m, b):

        generation = sum(
            m.P[g]
            for g in gens_at_bus[b]
        )

        outgoing = sum(
            m.flow[l]
            for l in lines_from_bus[b]
        )

        incoming = sum(
            m.flow[l]
            for l in lines_to_bus[b]
        )

        return (
            generation
            - demand_scenario[b]
            ==
            outgoing
            - incoming
        )

    m.nodal_balance = pyo.Constraint(
        m.BUSES,
        rule=nodal_balance_rule
    )

    # ----------------------------
    # Objective
    # ----------------------------

    m.objective = pyo.Objective(
        expr=sum(
            gen_cost[g] * m.P[g]
            for g in GEN_IDS
        ),
        sense=pyo.minimize
    )

    # ----------------------------
    # Duals
    # ----------------------------

    m.dual = pyo.Suffix(
        direction=pyo.Suffix.IMPORT
    )

    # ----------------------------
    # Solve
    # ----------------------------

    solver = pyo.SolverFactory("highs")

    results = solver.solve(
        m,
        tee=False
    )

    return m, results

In [ ]:
test_model, test_results = solve_rts_opf(
    demand_scenario=demand,
    gen_pmax_scenario=gen_pmax
)

print(
    "Status:",
    test_results.solver.status
)

print(
    "Termination:",
    test_results.solver.termination_condition
)

test_generation = sum(
    pyo.value(test_model.P[g])
    for g in GEN_IDS
)

test_cost = pyo.value(
    test_model.objective
)

print(
    f"Generation: {test_generation:,.6f} MW"
)

print(
    f"Demand:     {total_demand:,.6f} MW"
)

print(
    f"Cost:       ${test_cost:,.2f}/h"
)

In [ ]:
stress_results = []

stress_line_results = {}

for alpha in stress_levels:

    stressed_demand = {
        b: demand[b] * (1 + alpha)
        for b in BUS_IDS
    }

    m, res = solve_rts_opf(
        demand_scenario=stressed_demand,
        gen_pmax_scenario=gen_pmax
    )

    termination = (
        res.solver.termination_condition
    )

    if termination != pyo.TerminationCondition.optimal:

        stress_results.append(
            {
                "Stress": alpha,
                "Demand MW": sum(
                    stressed_demand.values()
                ),
                "Status": str(termination),
                "Cost $/h": np.nan,
                "Max Line Utilization": np.nan,
                "Binding Lines": np.nan
            }
        )

        continue

    scenario_flows = {
        l: pyo.value(m.flow[l])
        for l in LINE_IDS
    }

    scenario_utilization = {
        l:
        abs(scenario_flows[l])
        /
        line_limit[l]

        for l in LINE_IDS
    }

    max_utilization = max(
        scenario_utilization.values()
    )

    binding = [
        l
        for l in LINE_IDS
        if scenario_utilization[l]
        >= 0.999999
    ]

    stress_results.append(
        {
            "Stress": alpha,

            "Demand MW": sum(
                stressed_demand.values()
            ),

            "Status": str(termination),

            "Cost $/h": pyo.value(
                m.objective
            ),

            "Max Line Utilization":
                max_utilization,

            "Binding Lines":
                len(binding)
        }
    )

    stress_line_results[alpha] = {
        "model": m,
        "flows": scenario_flows,
        "utilization": scenario_utilization,
        "binding": binding
    }

In [ ]:
stress_summary = pd.DataFrame(
    stress_results
)

stress_summary

In [ ]:
bottleneck_rows = []

for alpha, scenario in stress_line_results.items():

    most_utilized_line = max(
        LINE_IDS,
        key=lambda l:
        scenario["utilization"][l]
    )

    bottleneck_rows.append(
        {
            "Stress": alpha,

            "Line":
                most_utilized_line,

            "From Bus":
                line_from[
                    most_utilized_line
                ],

            "To Bus":
                line_to[
                    most_utilized_line
                ],

            "Flow MW":
                scenario["flows"][
                    most_utilized_line
                ],

            "Limit MW":
                line_limit[
                    most_utilized_line
                ],

            "Utilization":
                scenario["utilization"][
                    most_utilized_line
                ]
        }
    )

bottleneck_summary = pd.DataFrame(
    bottleneck_rows
)

bottleneck_summary

In [ ]:
binding_detail_rows = []

for alpha, scenario in stress_line_results.items():

    m = scenario["model"]

    for l in scenario["binding"]:

        upper_dual = (
            m.dual[m.line_upper[l]]
            if m.line_upper[l] in m.dual
            else np.nan
        )

        lower_dual = (
            m.dual[m.line_lower[l]]
            if m.line_lower[l] in m.dual
            else np.nan
        )

        binding_detail_rows.append(
            {
                "Stress": alpha,
                "Line": l,
                "From Bus": line_from[l],
                "To Bus": line_to[l],
                "Flow MW":
                    scenario["flows"][l],
                "Limit MW":
                    line_limit[l],
                "Upper Dual":
                    upper_dual,
                "Lower Dual":
                    lower_dual
            }
        )

binding_details = pd.DataFrame(
    binding_detail_rows
)

binding_details

In [ ]:
baseline_stress_cost = (
    stress_summary.loc[
        stress_summary["Stress"] == 0,
        "Cost $/h"
    ].iloc[0]
)

stress_summary[
    "Cost Increase $/h"
] = (
    stress_summary["Cost $/h"]
    -
    baseline_stress_cost
)

stress_summary[
    "Added Load MW"
] = (
    stress_summary["Demand MW"]
    -
    total_demand
)

stress_summary[
    "Average Cost of Added Load $/MWh"
] = (
    stress_summary[
        "Cost Increase $/h"
    ]
    /
    stress_summary[
        "Added Load MW"
    ]
)

stress_summary.loc[
    stress_summary["Stress"] == 0,
    "Average Cost of Added Load $/MWh"
] = np.nan

stress_summary

In [ ]:
a27_rows = []

for alpha, scenario in stress_line_results.items():

    l = "A27"

    a27_rows.append(
        {
            "Stress": alpha,

            "Demand MW":
                total_demand
                * (1 + alpha),

            "A27 Flow MW":
                scenario["flows"][l],

            "A27 Absolute Flow MW":
                abs(
                    scenario["flows"][l]
                ),

            "A27 Limit MW":
                line_limit[l],

            "A27 Utilization":
                scenario[
                    "utilization"
                ][l],

            "A27 Headroom MW":
                line_limit[l]
                -
                abs(
                    scenario["flows"][l]
                )
        }
    )

a27_stress = pd.DataFrame(
    a27_rows
)

a27_stress

## 19. Diagnosing the Load-Stress Dispatch Transition

The load-stress experiment produced a non-monotonic transmission response.

As system demand increased from the baseline to +2%, flow magnitude on the most highly utilized branch, A27, decreased. Between +2% and +5%, however, A27 flow increased again and approached its thermal limit.

The average incremental production cost also increased slightly at +5%, suggesting that the optimal generation dispatch crossed an economic breakpoint.

This section compares generator dispatch across stress scenarios to identify the source of this transition.

In [ ]:
stress_dispatch = {}

for alpha, scenario in stress_line_results.items():

    m = scenario["model"]

    stress_dispatch[alpha] = {
        g: pyo.value(m.P[g])
        for g in GEN_IDS
    }

In [ ]:
dispatch_change_2_to_5 = (
    dispatch_results[
        [
            "GEN UID",
            "Bus ID",
            "Fuel",
            "Marginal Cost $/MWh"
        ]
    ]
    .copy()
)

dispatch_change_2_to_5["Dispatch +2% MW"] = (
    dispatch_change_2_to_5["GEN UID"]
    .map(stress_dispatch[0.02])
)

dispatch_change_2_to_5["Dispatch +5% MW"] = (
    dispatch_change_2_to_5["GEN UID"]
    .map(stress_dispatch[0.05])
)

dispatch_change_2_to_5["Change MW"] = (
    dispatch_change_2_to_5["Dispatch +5% MW"]
    -
    dispatch_change_2_to_5["Dispatch +2% MW"]
)

In [ ]:
dispatch_change_2_to_5[
    dispatch_change_2_to_5["Change MW"].abs()
    > 1e-6
].sort_values(
    "Change MW",
    ascending=False
)

In [ ]:
fuel_transition = (
    dispatch_change_2_to_5
    .groupby("Fuel")
    .agg(
        Dispatch_2pct_MW=(
            "Dispatch +2% MW",
            "sum"
        ),
        Dispatch_5pct_MW=(
            "Dispatch +5% MW",
            "sum"
        ),
        Change_MW=(
            "Change MW",
            "sum"
        )
    )
)

fuel_transition.sort_values(
    "Change_MW",
    ascending=False
)

In [ ]:
gen_capacity_transition = (
    dispatch_results[
        [
            "GEN UID",
            "Bus ID",
            "Fuel",
            "Available PMax MW",
            "Marginal Cost $/MWh"
        ]
    ]
    .copy()
)

gen_capacity_transition["Dispatch +2% MW"] = (
    gen_capacity_transition["GEN UID"]
    .map(stress_dispatch[0.02])
)

gen_capacity_transition["Dispatch +5% MW"] = (
    gen_capacity_transition["GEN UID"]
    .map(stress_dispatch[0.05])
)

gen_capacity_transition["Headroom +2% MW"] = (
    gen_capacity_transition["Available PMax MW"]
    -
    gen_capacity_transition["Dispatch +2% MW"]
)

gen_capacity_transition["Headroom +5% MW"] = (
    gen_capacity_transition["Available PMax MW"]
    -
    gen_capacity_transition["Dispatch +5% MW"]
)

gen_capacity_transition[
    (
        gen_capacity_transition[
            "Headroom +5% MW"
        ] <= 1e-6
    )
    &
    (
        gen_capacity_transition[
            "Available PMax MW"
        ] > 0
    )
].sort_values(
    "Marginal Cost $/MWh",
    ascending=False
).head(30)

In [ ]:
fine_stress_levels = np.arange(
    0.040,
    0.0701,
    0.0025
)

fine_results = []

fine_models = {}

for alpha in fine_stress_levels:

    stressed_demand = {
        b: demand[b] * (1 + alpha)
        for b in BUS_IDS
    }

    m, res = solve_rts_opf(
        demand_scenario=stressed_demand,
        gen_pmax_scenario=gen_pmax
    )

    termination = (
        res.solver.termination_condition
    )

    if termination != pyo.TerminationCondition.optimal:

        fine_results.append(
            {
                "Stress": alpha,
                "Demand MW":
                    sum(stressed_demand.values()),
                "Status": str(termination)
            }
        )

        continue

    scenario_flows = {
        l: pyo.value(m.flow[l])
        for l in LINE_IDS
    }

    utilization = {
        l:
        abs(scenario_flows[l])
        / line_limit[l]
        for l in LINE_IDS
    }

    max_line = max(
        LINE_IDS,
        key=lambda l: utilization[l]
    )

    binding = [
        l for l in LINE_IDS
        if utilization[l] >= 0.999999
    ]

    fine_results.append(
        {
            "Stress": alpha,
            "Demand MW":
                sum(stressed_demand.values()),

            "Cost $/h":
                pyo.value(m.objective),

            "Most Utilized Line":
                max_line,

            "Max Utilization":
                utilization[max_line],

            "A27 Flow MW":
                scenario_flows["A27"],

            "A27 Utilization":
                utilization["A27"],

            "Binding Lines":
                len(binding)
        }
    )

    fine_models[alpha] = m

fine_summary = pd.DataFrame(
    fine_results
)

fine_summary

In [ ]:
first_binding = fine_summary[
    fine_summary["Binding Lines"] > 0
]

if len(first_binding) > 0:

    first_binding_row = (
        first_binding.iloc[0]
    )

    print(
        "First tested stress level "
        "with binding transmission:"
    )

    display(
        first_binding_row
    )

else:

    print(
        "No transmission constraint "
        "became binding in the tested range."
    )

## 20. Critical Transmission-Congestion Threshold

The coarse and fine load-stress experiments show that A27 is the first transmission branch to become binding.

At a system-wide load increase of 5.25%, A27 remains slightly below its 500 MW thermal rating. At 5.50%, its flow reaches the thermal limit and remains constrained as demand increases.

The critical load-stress threshold is therefore defined as

$$
\alpha^*
=
\min\left\{
\alpha:
|f_{\mathrm{A27}}(\alpha)|
=
500\text{ MW}
\right\}.
$$

A numerical search is used to estimate this threshold more precisely.

In [ ]:
low = 0.0525
high = 0.0550

threshold_history = []

for iteration in range(20):

    mid = (low + high) / 2

    stressed_demand = {
        b: demand[b] * (1 + mid)
        for b in BUS_IDS
    }

    m, res = solve_rts_opf(
        demand_scenario=stressed_demand,
        gen_pmax_scenario=gen_pmax
    )

    assert (
        res.solver.termination_condition
        == pyo.TerminationCondition.optimal
    )

    a27_flow = pyo.value(
        m.flow["A27"]
    )

    a27_abs_flow = abs(
        a27_flow
    )

    a27_headroom = (
        line_limit["A27"]
        - a27_abs_flow
    )

    threshold_history.append(
        {
            "Iteration": iteration + 1,
            "Stress": mid,
            "Demand MW":
                sum(stressed_demand.values()),
            "A27 Flow MW":
                a27_flow,
            "A27 Headroom MW":
                a27_headroom
        }
    )

    # If still below the limit,
    # increase the stress.
    if a27_headroom > 1e-6:
        low = mid

    else:
        high = mid


threshold_history = pd.DataFrame(
    threshold_history
)

threshold_history.tail(10)

In [ ]:
critical_stress = high

critical_demand = (
    total_demand
    * (1 + critical_stress)
)

added_load_to_congestion = (
    critical_demand
    - total_demand
)

print(
    f"Estimated critical stress: "
    f"{critical_stress:.6%}"
)

print(
    f"Critical system demand: "
    f"{critical_demand:,.3f} MW"
)

print(
    f"Additional load from baseline: "
    f"{added_load_to_congestion:,.3f} MW"
)

In [ ]:
critical_demand_dict = {
    b:
    demand[b]
    * (1 + critical_stress)

    for b in BUS_IDS
}

critical_model, critical_results = (
    solve_rts_opf(
        demand_scenario=
            critical_demand_dict,

        gen_pmax_scenario=
            gen_pmax
    )
)

critical_line_rows = []

for l in LINE_IDS:

    flow = pyo.value(
        critical_model.flow[l]
    )

    utilization = (
        abs(flow)
        / line_limit[l]
    )

    critical_line_rows.append(
        {
            "Line": l,
            "From Bus": line_from[l],
            "To Bus": line_to[l],
            "Flow MW": flow,
            "Limit MW": line_limit[l],
            "Utilization": utilization
        }
    )

critical_lines = pd.DataFrame(
    critical_line_rows
)

critical_lines.sort_values(
    "Utilization",
    ascending=False
).head(10)

## 21. Marginal Economic Value of A27 Transmission Capacity

At a +6% system-load stress level, A27 is thermally constrained at its 500 MW rating.

The economic value of additional transmission capacity can therefore be estimated using a finite-difference experiment.

Let

$$
C(F)
$$

denote optimal operating cost when A27 has thermal capacity $F$.

The marginal value of additional A27 capacity is approximated by

$$
-\frac{\Delta C}{\Delta F}.
$$

This provides a numerical check against the optimization constraint dual.

In [ ]:
stress_for_capacity_test = 0.06

demand_6pct = {
    b:
    demand[b]
    * (1 + stress_for_capacity_test)

    for b in BUS_IDS
}

In [ ]:
model_500, results_500 = solve_rts_opf(
    demand_scenario=demand_6pct,
    gen_pmax_scenario=gen_pmax
)

cost_500 = pyo.value(
    model_500.objective
)

flow_500 = pyo.value(
    model_500.flow["A27"]
)

print(
    f"A27 capacity: 500 MW"
)

print(
    f"A27 flow: {flow_500:.6f} MW"
)

print(
    f"System cost: ${cost_500:,.6f}/h"
)

In [ ]:
line_limit_501 = (
    line_limit.copy()
)

line_limit_501["A27"] = 501.0

model_501, results_501 = solve_rts_opf(
    demand_scenario=demand_6pct,
    gen_pmax_scenario=gen_pmax,
    line_limit_scenario=line_limit_501
)

cost_501 = pyo.value(
    model_501.objective
)

flow_501 = pyo.value(
    model_501.flow["A27"]
)

capacity_value_fd = (
    cost_500
    - cost_501
)

print(
    f"C(500): ${cost_500:,.6f}/h"
)

print(
    f"C(501): ${cost_501:,.6f}/h"
)

print(
    f"Cost reduction: "
    f"${capacity_value_fd:,.6f}/h"
)

print(
    f"Finite-difference capacity value: "
    f"${capacity_value_fd:,.6f}/MW-h"
)

print(
    f"New A27 flow: "
    f"{flow_501:.6f} MW"
)

In [ ]:
upper_dual = (
    model_500.dual[
        model_500.line_upper["A27"]
    ]
)

lower_dual = (
    model_500.dual[
        model_500.line_lower["A27"]
    ]
)

print(
    "A27 upper-limit dual:",
    upper_dual
)

print(
    "A27 lower-limit dual:",
    lower_dual
)

print(
    "Finite-difference value:",
    capacity_value_fd
)

In [ ]:
a27_capacity_levels = [
    500,
    501,
    505,
    510,
    525,
    550,
    600
]

capacity_sweep_rows = []

for capacity in a27_capacity_levels:

    scenario_limits = (
        line_limit.copy()
    )

    scenario_limits[
        "A27"
    ] = capacity

    m, res = solve_rts_opf(
        demand_scenario=demand_6pct,
        gen_pmax_scenario=gen_pmax,
        line_limit_scenario=scenario_limits
    )

    cost = pyo.value(
        m.objective
    )

    flow = pyo.value(
        m.flow["A27"]
    )

    capacity_sweep_rows.append(
        {
            "A27 Capacity MW":
                capacity,

            "A27 Flow MW":
                flow,

            "A27 Utilization":
                abs(flow) / capacity,

            "Cost $/h":
                cost,

            "Savings vs 500 MW $/h":
                cost_500 - cost
        }
    )

a27_capacity_sweep = pd.DataFrame(
    capacity_sweep_rows
)

a27_capacity_sweep

## 22. Economically Useful A27 Capacity

The +6% load-stress scenario shows that relaxing A27 reduces operating cost only over a limited range of transmission capacity.

Although the branch can technically be assigned substantially larger thermal ratings, optimal flow stops increasing once sufficient capacity exists to support the unconstrained economic dispatch.

This distinguishes:

$$
\text{technical capacity}
$$

from

$$
\text{economically useful capacity}.
$$

The following experiment estimates the A27 capacity at which additional thermal capability ceases to reduce modeled operating cost.

In [ ]:
line_limit_unconstrained_a27 = (
    line_limit.copy()
)

line_limit_unconstrained_a27[
    "A27"
] = 5000.0

model_unconstrained_a27, results_unconstrained_a27 = (
    solve_rts_opf(
        demand_scenario=demand_6pct,
        gen_pmax_scenario=gen_pmax,
        line_limit_scenario=
            line_limit_unconstrained_a27
    )
)

unconstrained_a27_flow = abs(
    pyo.value(
        model_unconstrained_a27.flow["A27"]
    )
)

unconstrained_a27_cost = pyo.value(
    model_unconstrained_a27.objective
)

print(
    f"Economically desired A27 flow: "
    f"{unconstrained_a27_flow:.6f} MW"
)

print(
    f"Cost with A27 unconstrained: "
    f"${unconstrained_a27_cost:,.6f}/h"
)

print(
    f"Maximum operating-cost savings: "
    f"${cost_500 - unconstrained_a27_cost:,.6f}/h"
)

In [ ]:
economically_useful_uplift = (
    unconstrained_a27_flow
    - line_limit["A27"]
)

print(
    f"Baseline A27 capacity: "
    f"{line_limit['A27']:.3f} MW"
)

print(
    f"Economically useful capacity: "
    f"{unconstrained_a27_flow:.3f} MW"
)

print(
    f"Economically useful uplift: "
    f"{economically_useful_uplift:.3f} MW"
)

print(
    f"Useful uplift percentage: "
    f"{100 * economically_useful_uplift / line_limit['A27']:.3f}%"
)

## 23. Notebook 3 Conclusions

This notebook developed and independently validated a single-period DC optimal power flow model for the RTS-GMLC system.

For the selected DAY_AHEAD peak-load hour:

- system demand is approximately 8,191.84 MW;
- modeled operating cost is approximately $130,429.23/h;
- no transmission thermal constraint binds in the baseline solution;
- all available wind and solar generation is dispatched;
- A27 is the most highly utilized branch at approximately 99.25%.

A proportional load-stress experiment shows that A27 becomes the first binding transmission constraint at approximately a 5.289% increase in system demand, corresponding to approximately 8,625.09 MW of total load.

At a +6% load-stress state, A27 is binding at its 500 MW thermal rating. Increasing its capacity by 1 MW reduces modeled operating cost by approximately $5.84/h. This finite-difference value agrees with the magnitude of the corresponding optimization dual.

However, the economic value of additional A27 capacity is not constant. Once sufficient capacity is provided to accommodate the economically desired flow, further thermal capacity produces no additional operating-cost savings in that state.

These results establish the single-period optimization and sensitivity framework required for subsequent chronological analysis. They do not establish A27 as the final reconductoring corridor or represent the thesis's principal empirical result.